# Coverage-Only Ablation: Proving GP Adds Nothing

**Critical experiment for NeurIPS submission.**

**Hypothesis:** If coverage-only ≈ CAGP, then GP component is worthless.

```
Coverage-only:  U = 2 - cov[h,r] - cov[t,r]  (no learning)
CAGP:           U = 0.5 * GP_var + 0.5 * coverage  (learned + explicit)
```

If both achieve ~0.96 AUROC, **the GP machinery adds nothing**.

In [ ]:
!pip install -q torch numpy scikit-learn pykeen

In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
def load_dataset(name):
    """Load dataset via PyKEEN."""
    if name == 'FB15k-237':
        from pykeen.datasets import FB15k237
        dataset = FB15k237(create_inverse_triples=False)
    elif name == 'WN18RR':
        from pykeen.datasets import WN18RR
        dataset = WN18RR(create_inverse_triples=False)
    elif name == 'YAGO3-10':
        from pykeen.datasets import YAGO310
        dataset = YAGO310(create_inverse_triples=False)
    else:
        raise ValueError(f"Unknown dataset: {name}")
    
    def extract(tf):
        return [(h, r, t) for h, r, t in tf.triples]
    
    train = extract(dataset.training)
    test = extract(dataset.testing)
    
    entities = set()
    relations = set()
    for h, r, t in train + test:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    return {
        'train': train, 'test': test,
        'entities': list(entities), 'relations': list(relations)
    }

In [ ]:
class CoverageOnlyDetector:
    """
    No learning. Just tracks coverage.
    
    U = 2 - coverage[head, relation] - coverage[tail, relation]
    """
    def __init__(self, num_entities, num_relations):
        self.coverage = torch.zeros(num_entities, num_relations)
    
    def fit(self, triples, entity_to_idx, relation_to_idx):
        """Build coverage matrix from training triples."""
        for h, r, t in triples:
            h_idx = entity_to_idx[h]
            r_idx = relation_to_idx[r]
            t_idx = entity_to_idx[t]
            self.coverage[h_idx, r_idx] = 1.0
            self.coverage[t_idx, r_idx] = 1.0
        return self
    
    def get_uncertainty(self, heads, relations, tails):
        """Pure coverage-based uncertainty."""
        h_seen = self.coverage[heads, relations]
        t_seen = self.coverage[tails, relations]
        return 2.0 - h_seen - t_seen


def evaluate_auroc(detector, test_triples, entity_to_idx, relation_to_idx, num_entities):
    """Evaluate AUROC for OOD detection."""
    heads = torch.tensor([entity_to_idx.get(h, 0) for h, r, t in test_triples])
    relations = torch.tensor([relation_to_idx.get(r, 0) for h, r, t in test_triples])
    tails = torch.tensor([entity_to_idx.get(t, 0) for h, r, t in test_triples])
    
    # ID uncertainty
    id_unc = detector.get_uncertainty(heads, relations, tails).numpy()
    
    # OOD uncertainty (random tails)
    neg_tails = torch.randint(0, num_entities, tails.shape)
    ood_unc = detector.get_uncertainty(heads, relations, neg_tails).numpy()
    
    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])  # Lower uncertainty = higher confidence
    
    return roc_auc_score(labels, scores)

## Run on All Datasets

In [ ]:
DATASETS = ['WN18RR', 'FB15k-237']  # Add 'YAGO3-10' if you have time
SEEDS = [42, 123, 456]

results = {}

for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}")
    print('='*60)
    
    data = load_dataset(dataset_name)
    print(f"Train: {len(data['train'])}, Test: {len(data['test'])}")
    print(f"Entities: {len(data['entities'])}, Relations: {len(data['relations'])}")
    
    ent2idx = {e: i for i, e in enumerate(data['entities'])}
    rel2idx = {r: i for i, r in enumerate(data['relations'])}
    
    aurocs = []
    for seed in SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # Fit coverage detector (no training, just counting)
        detector = CoverageOnlyDetector(len(ent2idx), len(rel2idx))
        detector.fit(data['train'], ent2idx, rel2idx)
        
        # Evaluate
        auroc = evaluate_auroc(detector, data['test'], ent2idx, rel2idx, len(ent2idx))
        aurocs.append(auroc)
        print(f"  Seed {seed}: AUROC = {auroc:.4f}")
    
    results[dataset_name] = {
        'mean': np.mean(aurocs),
        'std': np.std(aurocs),
        'relations': len(data['relations'])
    }
    print(f"  => Mean: {results[dataset_name]['mean']:.4f} ± {results[dataset_name]['std']:.4f}")

## Comparison Table

In [ ]:
# Previous results (from CAGP experiments)
CAGP_RESULTS = {
    'WN18RR': {'mean': 0.871, 'std': 0.001},
    'FB15k-237': {'mean': 0.960, 'std': 0.000},
}

print("\n" + "="*70)
print("COVERAGE-ONLY vs CAGP COMPARISON")
print("="*70)
print(f"{'Dataset':<15} {'Relations':<10} {'Coverage-Only':<18} {'CAGP':<18} {'Diff'}")
print("-"*70)

for name in results:
    cov = results[name]
    cagp = CAGP_RESULTS.get(name, {'mean': 0, 'std': 0})
    diff = cov['mean'] - cagp['mean']
    
    print(f"{name:<15} {cov['relations']:<10} {cov['mean']:.4f} ± {cov['std']:.3f}    "
          f"{cagp['mean']:.4f} ± {cagp['std']:.3f}    {diff:+.4f}")

print("-"*70)
print("\nINTERPRETATION:")
print("If Coverage-Only ≈ CAGP, then GP component adds NOTHING.")
print("This proves the negative result: complex GP methods are unnecessary.")

In [ ]:
# Save results
import json

output = {
    'experiment': 'coverage_only_ablation',
    'purpose': 'Prove GP component adds nothing beyond coverage',
    'results': results,
    'comparison_to_cagp': CAGP_RESULTS,
    'conclusion': 'If coverage-only matches CAGP, GP is unnecessary'
}

with open('coverage_only_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to coverage_only_results.json")